In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.query import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *

In [3]:
import pandas as pd
customer_info_path = _root / "init" / "CustomerInfo.csv"
customer_info_df = pd.read_csv(customer_info_path)

In [4]:
customers = KPI_Customer.query_table({'db_path': DB_PATH})


In [ ]:
# Default utilization for every customer in KPI_Customer
for _, customer in customers.iterrows():
    add_customer_utilization(customer['Name'])

# Overwrite with CustomerInfo.csv where values are provided

for _, customer in customer_info_df.iterrows():
    print(customer['CustomerName'])
    base_surveyors = customer['BaseSurveyors'] if pd.notna(customer['BaseSurveyors']) else 0

    add_customer_utilization(
        customer['CustomerName'],
        customer['WorkingHours'],
        customer['WorkingDays'],
        customer['BaseSurveyors'],
        customer['SunriseHour'],
        customer['SunsetHour'],
        base_surveyors,
    )

    current_year = datetime.now().year
    default_por_start_date = f'01-01-{current_year}'
    default_por_end_date = f'31-12-{current_year}'
    add_por(
        customer_name = customer['CustomerName'],
        year = 2026,
        value = customer['POR'] if pd.notna(customer['POR']) else 0,
        StartingDate = customer['PORStartingDate'] if pd.notna(customer['PORStartingDate']) else default_por_start_date,
        EndingDate = customer['POREndingDate'] if pd.notna(customer['POREndingDate']) else default_por_end_date,
        Description = ''
   
    )
    print('POR added')
    KPI_POR.query_table({'db_path': DB_PATH})
    add_output_excel_location(customer['CustomerName'], customer['OutputExcelFolder'], KPIHub_Conn)
    KPI_OutputExcelLocation.query_table({'db_path': DB_PATH}) 
    print('Output Excel Location added')



In [6]:
KPI_Customer.query_table({'db_path': DB_PATH})

In [7]:
KPI_POR.query_table({'db_path': DB_PATH})

In [8]:
KPI_Utilization.query_table({'db_path': DB_PATH})